# Image classification with CLIP

In this notebook, we explore how to use CLIP to classify images using zero-shot classification.
We will apply the following steps:
* Install required packages
* Load a dataset (CIFAR10 here)
* Classify all the images of the dataset
* Compute several metrics and display the confusion matrix

## Install required packages

In [ ]:
#Restart session after running this
!pip install -U fsspec==2023.6.0
!pip install transformers torch datasets matplotlib scikit-learn

## Load CLIP: model, processor and tokenizer using transformers
Firstly, import the required packages.
Then, define the device to use (i.e gpu or cpu).
Lastly, load CLIP model, processor and tokenizer using Transformers.

In [ ]:
#Import packages
import torch
from PIL import Image
from transformers import AutoProcessor, CLIPModel, AutoTokenizer
import tensorflow as tf
from tensorflow.keras.datasets import cifar10
from datasets import load_dataset
from tqdm import tqdm

#Define device (GPU or CPU)
device = torch.device('cuda' if torch.cuda.is_available() else "cpu")

#Load CLIP model, processor and tokenizer
processor = AutoProcessor.from_pretrained("openai/clip-vit-base-patch32")
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
tokenizer = AutoTokenizer.from_pretrained("openai/clip-vit-base-patch32")

## Load CIFAR 10 dataset
We use this dataset as it has 10 pre-defined classes.

In [ ]:
# Clear only the contents inside the datasets cache
!rm -rf ~/.cache/huggingface/datasets/*

# Load the datasets library
from datasets import load_dataset

# Load CIFAR-10 dataset
dataset = load_dataset("cifar10")

# Display the list of labels
labels = dataset["train"].features["label"].names
print(labels)

## Classification of one image
Firstly, we are going to classify an image using CLIP to see how it works.
We define a function named "classify", which takes as input an image and a list of labels, and returns the label with the highest classification score.

In [ ]:
#Take the first image in the training set
image = dataset['train'][0]['img']

#Function to classify an image among the list of labels
def classify(image, labels):
    inputs = processor(text=labels, images=image, return_tensors="pt", padding=True).to(device)
    outputs = model(**inputs)
    logits_per_image = outputs.logits_per_image
    probs = logits_per_image.softmax(dim=1)
    label = probs.argmax()
    return label

l = classify(image,labels)
#Display the first image in the dataset and its predicted class (i.e. airplane, which is correct)
display(image)
print(labels[l])

## Classification of all the images in the test set (10k images)
We are now going to use the same method on the test set of the CIFAR dataset.
We create a variable named "predictions", which will store the predicted label for each image. In order to compute several metrics afterwards, we also store the ground truth.
First step: we iterate over the dataset, classify all the images and store the predicted labels.

In [ ]:
predictions= []
ground_truth= [d['label'] for d in dataset['test']]

for img in tqdm(dataset['test']):
    pred = classify(img['img'], labels)
    predictions.append(pred.item())

In order to assess the performance of the classification, we compute a set of metrics, using the functions available in the sklearn module.
We will more particularly look into: accuracy, precision, recall and f1-score. We also compute the confusion matrix to get insights into what classes were misclassified.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
y_true = ground_truth
y_pred = predictions

# Compute accuracy
accuracy = accuracy_score(y_true, y_pred)
print(f'Accuracy: {accuracy:.4f}')

precision = precision_score(y_true, y_pred, average='weighted')
print(f'Precision: {precision:.4f}')

recall = recall_score(y_true, y_pred, average='weighted')
print(f'Recall: {recall:.4f}')

f1 = f1_score(y_true, y_pred, average='weighted')
print(f'F1 Score: {f1:.4f}')

# Generate classification report
print('\nClassification Report:')
print(classification_report(y_true, y_pred))

# Generate confusion matrix
conf_matrix = confusion_matrix(y_true, y_pred)
print('\nConfusion Matrix:')
print(conf_matrix)

We plot the confusion matrix to make it more human readable.

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

disp = ConfusionMatrixDisplay(confusion_matrix=conf_matrix,
                              display_labels=labels)
disp.plot()
plt.show()

## Analysis
CLIP demonstrated a commendable accuracy at 87.3%.
The confusion matrix reveals lower recall rates for frogs and cats. Frogs were frequently misclassified as birds, while cats were commonly misclassified as dogs.
The upcoming section delves into prompt engineering, offering insights into potential enhancements for the achieved scores.


The [CLIP paper](https://arxiv.org/pdf/2103.00020.pdf) explains that it is relatively rare in the training set to encounter an image being described with a unique word. Therefore, adding "a photo of" before the label improved accuracy by 1.3% on ImageNet.
We will experiment with it here and assess whether we observe any improvements.

In [ ]:
#Define new labels
new_labels = [f"a photo of {label}" for label in labels]
print(new_labels)

Reusing the code above, we get:

In [ ]:
predictions= []
ground_truth= [d['label'] for d in dataset['test']]

for img in tqdm(dataset['test']):
    pred = classify(img['img'], new_labels)
    predictions.append(pred.item())

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay # Ensure ConfusionMatrixDisplay is imported
import matplotlib.pyplot as plt
y_true = ground_truth
y_pred = predictions

# Compute accuracy
accuracy = accuracy_score(y_true, y_pred)
print(f'Accuracy: {accuracy:.4f}')

precision = precision_score(y_true, y_pred, average='weighted')
print(f'Precision: {precision:.4f}')

recall = recall_score(y_true, y_pred, average='weighted')
print(f'Recall: {recall:.4f}')

f1 = f1_score(y_true, y_pred, average='weighted')
print(f'F1 Score: {f1:.4f}')

# Generate classification report
print('\nClassification Report:')
print(classification_report(y_true, y_pred))

# Generate confusion matrix
conf_matrix = confusion_matrix(y_true, y_pred)
print('\nConfusion Matrix:')
print(conf_matrix)

#With plot only labels and  not new_label for the sake of visibility
disp = ConfusionMatrixDisplay(confusion_matrix=conf_matrix,
                              display_labels=labels)
disp.plot()
plt.show()

With the application of straightforward prompt engineering, we successfully increased the accuracy score by 1.6%. To conclude, the fine-tuning of prompts has the potential to generate even more substantial improvements.